# 04 — SVR Modeling
## Peramalan Nilai PM2.5 Harian DKI Jakarta dengan SVR

Notebook ini adalah tahap akhir: baseline, **forward selection** untuk menentukan fitur final dari kandidat luas notebook 03, tuning SVR, evaluasi, analisis residual, dan eksperimen variabel polutan tambahan.

Langkah-langkah:
1. Muat `data/train.csv` dan `data/test.csv`
2. Baseline naive (persistence)
3. **Forward selection** dengan `TimeSeriesSplit` — memilih fitur final dari kandidat lag notebook 03 berdasarkan performa, bukan ambang ACF/PACF
4. Tuning hyperparameter SVR pada fitur yang terpilih
5. Evaluasi pada data uji, dibandingkan baseline
6. Analisis residual
7. Eksperimen: apakah menambah polutan lain (PM10, CO, O₃) memperbaiki akurasi?

> **Prinsip penting (lihat `docs/metodologi.md`):** fitur final ditentukan oleh performa model lewat forward selection, bukan oleh besaran ACF/PACF semata. ACF/PACF di notebook 02 hanya memberi kandidat; notebook ini yang memutuskan.

## 1. Setup dan Muat Data

In [ ]:
import os

if 'google.colab' in str(get_ipython()) and not os.path.exists('data'):
    # Ganti USERNAME dengan username GitHub Anda
    !git clone https://github.com/USERNAME/jakarta-pm25-svr-forecasting.git
    %cd jakarta-pm25-svr-forecasting

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)
RANDOM_STATE = 42

In [ ]:
train = pd.read_csv('data/train.csv', parse_dates=['tanggal']).set_index('tanggal')
test = pd.read_csv('data/test.csv', parse_dates=['tanggal']).set_index('tanggal')

print(f"Data latih: {len(train):,} baris ({train.index.min().date()} s.d. {train.index.max().date()})")
print(f"Data uji  : {len(test):,} baris ({test.index.min().date()} s.d. {test.index.max().date()})")

TARGET = 'pm25'
# Kandidat fitur dari pm25 saja (bukan polutan lain — itu dipakai khusus di eksperimen Bagian 7)
KANDIDAT_FITUR = [c for c in train.columns if c.startswith('pm25_lag') or c in ('pm25_roll7', 'pm25_roll14', 'month_sin', 'month_cos')]
print(f"\nJumlah kandidat fitur: {len(KANDIDAT_FITUR)}")
print(KANDIDAT_FITUR)

## 2. Baseline Naive (Persistence)

$$\hat{y}_t = y_{t-1}$$

Model SVR harus mengalahkan ini untuk dianggap berguna.

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mae(y_true, y_pred):
    return float(mean_absolute_error(y_true, y_pred))

def smape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return float(np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred))) * 100)

pred_naive = test['pm25_lag1']  # nilai kemarin sebagai prediksi hari ini

hasil_naive = {
    'Model': 'Naive (persistence)',
    'RMSE': rmse(test[TARGET], pred_naive),
    'MAE': mae(test[TARGET], pred_naive),
    'sMAPE (%)': smape(test[TARGET], pred_naive),
    'R2': r2_score(test[TARGET], pred_naive),
}
print(pd.Series(hasil_naive))

## 3. Forward Selection dengan Time Series Cross-Validation

**Prinsip:** mulai dari himpunan fitur kosong. Pada setiap iterasi, coba tambahkan satu fitur dari daftar kandidat yang paling menurunkan RMSE rata-rata pada `TimeSeriesSplit`. Berhenti jika penambahan fitur tidak lagi memperbaiki RMSE secara berarti (lihat `AMBANG_PERBAIKAN`).

Model yang dipakai selama seleksi adalah SVR dengan hyperparameter default yang masuk akal (bukan hasil tuning), karena tuning penuh di setiap kombinasi fitur akan sangat lambat. Tuning hyperparameter sesungguhnya dilakukan setelah fitur final terpilih (Bagian 4).

In [ ]:
def svr_cv_rmse(X, y, n_splits=5):
    """Rata-rata RMSE TimeSeriesSplit untuk SVR dengan hyperparameter default."""
    tscv = TimeSeriesSplit(n_splits=n_splits)
    skor = []
    for idx_train, idx_val in tscv.split(X):
        X_tr, X_val = X.iloc[idx_train], X.iloc[idx_val]
        y_tr, y_val = y.iloc[idx_train], y.iloc[idx_val]

        model = TransformedTargetRegressor(
            regressor=Pipeline([('scaler', StandardScaler()), ('svr', SVR(C=10, gamma='scale', epsilon=0.1))]),
            transformer=StandardScaler()
        )
        model.fit(X_tr, y_tr)
        pred = model.predict(X_val)
        skor.append(rmse(y_val, pred))
    return float(np.mean(skor))

In [ ]:
AMBANG_PERBAIKAN = 0.001  # fitur baru harus memperbaiki RMSE CV minimal sebesar ini (dalam satuan nilai ISPU)
MAKS_FITUR = 8            # batas atas jumlah fitur, agar seleksi tidak berjalan tanpa henti

fitur_terpilih = []
fitur_tersisa = list(KANDIDAT_FITUR)
riwayat_seleksi = []

rmse_terbaik_sejauh_ini = np.inf

for iterasi in range(1, MAKS_FITUR + 1):
    skor_kandidat = {}
    for fitur in fitur_tersisa:
        kombinasi = fitur_terpilih + [fitur]
        skor_kandidat[fitur] = svr_cv_rmse(train[kombinasi], train[TARGET])

    fitur_terbaik = min(skor_kandidat, key=skor_kandidat.get)
    rmse_terbaik_iterasi = skor_kandidat[fitur_terbaik]
    perbaikan = rmse_terbaik_sejauh_ini - rmse_terbaik_iterasi

    riwayat_seleksi.append({
        'iterasi': iterasi,
        'fitur_ditambahkan': fitur_terbaik,
        'rmse_cv': round(rmse_terbaik_iterasi, 4),
        'perbaikan': round(perbaikan, 4) if np.isfinite(perbaikan) else None,
    })

    print(f"Iterasi {iterasi}: +{fitur_terbaik:15s} RMSE CV = {rmse_terbaik_iterasi:.4f} "
          f"(perbaikan: {perbaikan:.4f})" if np.isfinite(perbaikan) else
          f"Iterasi {iterasi}: +{fitur_terbaik:15s} RMSE CV = {rmse_terbaik_iterasi:.4f}")

    if perbaikan < AMBANG_PERBAIKAN:
        print(f"\nBerhenti: penambahan fitur tidak lagi memperbaiki RMSE CV >= {AMBANG_PERBAIKAN}.")
        break

    fitur_terpilih.append(fitur_terbaik)
    fitur_tersisa.remove(fitur_terbaik)
    rmse_terbaik_sejauh_ini = rmse_terbaik_iterasi

print(f"\n>>> FITUR FINAL (dari forward selection): {fitur_terpilih}")

In [ ]:
riwayat_df = pd.DataFrame(riwayat_seleksi)
print(riwayat_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(riwayat_df['iterasi'], riwayat_df['rmse_cv'], marker='o')
ax.set_xlabel('Iterasi (jumlah fitur)')
ax.set_ylabel('RMSE rata-rata TimeSeriesSplit')
ax.set_title('Forward Selection: RMSE CV per Iterasi')
ax.set_xticks(riwayat_df['iterasi'])
plt.tight_layout()
plt.savefig('docs/forward_selection.png', dpi=120)
plt.show()

**Catatan penting:** `fitur_terpilih` adalah hasil nyata dari data Anda — bisa berbeda dari dugaan awal di notebook 02/03. Itulah intinya forward selection: lag yang tampak "signifikan" di PACF belum tentu benar-benar membantu performa model, dan sebaliknya.

## 4. Tuning Hyperparameter pada Fitur Final

Setelah fitur ditentukan, baru dilakukan pencarian grid untuk `C`, `epsilon`, `gamma` secara menyeluruh.

In [ ]:
param_grid = {
    'regressor__svr__kernel': ['rbf'],
    'regressor__svr__C': [1, 10, 100],
    'regressor__svr__gamma': ['scale', 0.01, 0.05, 0.1],
    'regressor__svr__epsilon': [0.01, 0.05, 0.1, 0.3],
}

base_model = TransformedTargetRegressor(
    regressor=Pipeline([('scaler', StandardScaler()), ('svr', SVR())]),
    transformer=StandardScaler()
)

tscv = TimeSeriesSplit(n_splits=5)
grid = GridSearchCV(base_model, param_grid, cv=tscv, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid.fit(train[fitur_terpilih], train[TARGET])

print("Hyperparameter terbaik:", grid.best_params_)
print(f"RMSE CV terbaik: {-grid.best_score_:.4f}")

model_final = grid.best_estimator_

## 5. Evaluasi pada Data Uji

In [ ]:
pred_svr = model_final.predict(test[fitur_terpilih])

hasil_svr = {
    'Model': 'SVR (fitur terpilih)',
    'RMSE': rmse(test[TARGET], pred_svr),
    'MAE': mae(test[TARGET], pred_svr),
    'sMAPE (%)': smape(test[TARGET], pred_svr),
    'R2': r2_score(test[TARGET], pred_svr),
}

tabel_hasil = pd.DataFrame([hasil_naive, hasil_svr]).round(3)
print(tabel_hasil.to_string(index=False))

perbaikan_rmse = (hasil_naive['RMSE'] - hasil_svr['RMSE']) / hasil_naive['RMSE'] * 100
print(f"\nSVR memperbaiki RMSE sebesar {perbaikan_rmse:.1f}% dibanding baseline naive."
      if perbaikan_rmse > 0 else
      f"\nSVR TIDAK lebih baik dari baseline naive (selisih RMSE: {-perbaikan_rmse:.1f}% lebih buruk).")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(test.index, test[TARGET], label='Aktual', linewidth=1)
ax.plot(test.index, pred_svr, label='Prediksi SVR', linewidth=1, alpha=0.8)
ax.plot(test.index, pred_naive, label='Prediksi Naive', linewidth=0.8, alpha=0.5, linestyle='--')
ax.set_title('Aktual vs Prediksi — Data Uji')
ax.set_ylabel('Nilai ISPU PM2.5')
ax.legend()
plt.tight_layout()
plt.savefig('docs/actual_vs_pred.png', dpi=120)
plt.show()

## 6. Analisis Residual

In [ ]:
residual = test[TARGET] - pred_svr

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(test.index, residual, linewidth=0.8)
axes[0].axhline(0, color='red', linewidth=0.8, linestyle='--')
axes[0].set_title('Residual terhadap Waktu')
axes[0].set_ylabel('Residual (aktual - prediksi)')

axes[1].hist(residual, bins=30, color='steelblue', edgecolor='white')
axes[1].set_title('Distribusi Residual')
axes[1].axvline(0, color='red', linewidth=0.8, linestyle='--')

plt.tight_layout()
plt.savefig('docs/residuals.png', dpi=120)
plt.show()

print(f"Rata-rata residual: {residual.mean():.3f} (idealnya mendekati 0)")
print(f"Std residual       : {residual.std():.3f}")
print(f"5 hari dengan residual absolut terbesar:")
print(residual.abs().sort_values(ascending=False).head())

## 7. Eksperimen: Apakah Polutan Lain Menambah Akurasi?

Menguji penambahan `pm10_lag1`, `co_lag1`, `o3_lag1` di atas fitur final yang sudah terpilih dari forward selection. Baris dengan nilai polutan kosong dibuang khusus untuk eksperimen ini (lihat `docs/dataset.md` bagian 6 soal celah panjang `pm10`), sehingga data latihnya lebih sedikit dibanding model utama.

In [ ]:
skenario_fitur = {
    'SVR (pm25 saja, fitur terpilih)': fitur_terpilih,
    'SVR + CO, O3': fitur_terpilih + ['co_lag1', 'o3_lag1'],
    'SVR + PM10': fitur_terpilih + ['pm10_lag1'],
    'SVR + semua polutan': fitur_terpilih + ['pm10_lag1', 'so2_lag1', 'co_lag1', 'o3_lag1', 'no2_lag1'],
}

hasil_eksperimen = [hasil_naive]

for nama, kolom_fitur in skenario_fitur.items():
    train_sub = train[kolom_fitur + [TARGET]].dropna()
    test_sub = test[kolom_fitur + [TARGET]].dropna()

    model = TransformedTargetRegressor(
        regressor=Pipeline([('scaler', StandardScaler()), ('svr', SVR(C=10, gamma='scale', epsilon=0.1))]),
        transformer=StandardScaler()
    )
    model.fit(train_sub[kolom_fitur], train_sub[TARGET])
    pred = model.predict(test_sub[kolom_fitur])

    hasil_eksperimen.append({
        'Model': nama,
        'RMSE': rmse(test_sub[TARGET], pred),
        'MAE': mae(test_sub[TARGET], pred),
        'sMAPE (%)': smape(test_sub[TARGET], pred),
        'R2': r2_score(test_sub[TARGET], pred),
        'n_latih': len(train_sub),
    })

tabel_eksperimen = pd.DataFrame(hasil_eksperimen)
print(tabel_eksperimen.round(3).to_string(index=False))

**Catatan:** model "SVR (fitur terpilih)" pada tabel di atas memakai hyperparameter default (`C=10, gamma='scale', epsilon=0.1`), BUKAN hasil tuning di Bagian 4, supaya perbandingan antar-skenario adil (semua pakai setting yang sama). Angka RMSE-nya karena itu mungkin sedikit berbeda dari Bagian 5. Kesimpulan "apakah polutan lain membantu" dilihat dari perbandingan **relatif** antar baris pada tabel ini.

## Ringkasan Hasil

Salin angka-angka berikut ke README setelah notebook ini dijalankan:

- Fitur final hasil forward selection: dicetak otomatis pada Bagian 3
- Hyperparameter terbaik: dicetak otomatis pada Bagian 4
- Tabel RMSE/MAE/sMAPE/R² (naive vs SVR): dicetak otomatis pada Bagian 5
- Tabel eksperimen polutan: dicetak otomatis pada Bagian 7

**Proyek selesai sampai di sini.** Langkah selanjutnya murni dokumentasi: salin hasil ke README, periksa `[ISI: ...]` yang masih tersisa di `docs/dataset.md` dan `docs/metodologi.md`, lalu unggah semua file final ke GitHub.